# Retrain the Fallacy Detector (DistilBERT, v2 data)

Fixes the model that was over-flagging everything. The old detector was trained **only** on
the balanced contrastive set (every example is an argument), so it had never seen ordinary
prose — it flagged **62% of clean real-world sentences** as fallacies ("Thank you both." →
0.99 fallacy).

**The fix:** retrain with **MAFALDA**'s real-world data, which contains 601 sentences
explicitly labelled *non*-fallacious. Those are the negatives the detector was missing.

Measured on held-out real-world prose (TF-IDF proof-of-concept):

| detector | precision | false alarms on clean prose |
|---|---|---|
| old (contrastive only) | 0.52 | **62%** |
| new (+ MAFALDA negatives) | 0.72 | **24%** |

This notebook trains **both stages** on the improved data and exports drop-in models.

---

### What to do
1. **Runtime → Change runtime type → T4 GPU**
2. Upload your **`data/`** folder into the Colab session (the file pane on the left — the
   same way you did before). It needs: `contrastive_dataset.csv`, `edu_train.csv`,
   `gold_standard_dataset.jsonl`.
3. **Runtime → Run all**
4. The last cell zips the models and downloads them. Unzip into
   `fallacysuspect/models/two_stage/` (replacing the old `*_distilbert/` folders).

## 0. Setup

In [ ]:
%pip install -q transformers torch scikit-learn pandas numpy

In [ ]:
import json, os, glob, time
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE.type != "cuda":
    print("  !! No GPU — Runtime > Change runtime type > T4 GPU (CPU works but is slow)")

# --- find the uploaded data/ folder in the Colab session ---
MARKERS = ["contrastive_dataset.csv", "gold_standard_dataset.jsonl"]
def has_data(d):
    return bool(d) and all(os.path.exists(os.path.join(d, m)) for m in MARKERS)

DATA_DIR = None
for c in ["data", "/content/data", "/content", ".", "../data"]:
    if has_data(c):
        DATA_DIR = c; break
if DATA_DIR is None:                      # last resort: search the session
    for m in MARKERS[:1]:
        hits = glob.glob(os.path.join("/content", "**", m), recursive=True)
        if hits:
            DATA_DIR = os.path.dirname(hits[0]); break

print("DATA_DIR:", DATA_DIR, "| found:", has_data(DATA_DIR))
assert has_data(DATA_DIR), "Upload your data/ folder into the session, then re-run this cell."

## 1. Config

In [ ]:
MODEL       = "distilbert-base-uncased"
MAX_LEN     = 96
BATCH       = 32          # GPU can handle a bigger batch than the CPU script
LR          = 3e-5
DET_EPOCHS  = 4           # detector (binary)
TYP_EPOCHS  = 6           # typer (14 classes, needs a bit more)
TEST_FRAC   = 0.2         # share of MAFALDA *documents* held out for the final score
VAL_FRAC    = 0.1         # ...and for picking the best epoch (never scored on)
MIN_WORDS   = 5
DATA_VERSION = 2          # v2 = trained with MAFALDA real-world negatives
OUT_DIR     = "/content/models/two_stage" if os.path.isdir("/content") else "models/two_stage"
os.makedirs(OUT_DIR, exist_ok=True)
print("models will be written to:", OUT_DIR)

## 1b. Pre-download DistilBERT (with retries)

The base weights are ~268 MB from Hugging Face. Grabbing them up front — with retry +
resume — means a flaky download can't masquerade as "training is stuck". Normally takes
seconds; if a run stalls, just re-run **this cell** (it resumes from cache).

In [ ]:
# Plain from_pretrained in a retry loop — no huggingface_hub kwargs (they change
# between versions; `resume_download` is deprecated/removed and would crash here).
# Downloads resume from cache automatically on retry.
for attempt in range(1, 6):
    try:
        t0 = time.time()
        AutoTokenizer.from_pretrained(MODEL)
        AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
        print(f"cached DistilBERT in {time.time()-t0:.0f}s — training will load instantly from here.")
        break
    except Exception as exc:
        print(f"attempt {attempt} failed ({type(exc).__name__}: {exc}) — retrying in 5s...")
        time.sleep(5)
else:
    raise RuntimeError(
        "Could not download DistilBERT after 5 tries. "
        "Runtime > Restart session, then Run all."
    )

## 2. Build the improved datasets

Merges the three sources, and splits MAFALDA **by document** so no sentence from a test
document leaks into training. Also adds **slippery slope**, which the LOGIC taxonomy lacks
entirely despite being a debate staple → **14 classes**.

In [ ]:
# MAFALDA's 23 fine-grained types -> our taxonomy
MAFALDA_TO_OURS = {
    "hasty generalization": "faulty generalization",
    "appeal to ridicule": "appeal to emotion",
    "slippery slope": "slippery slope",
    "causal oversimplification": "false causality",
    "ad hominem": "ad hominem",
    "false dilemma": "false dilemma",
    "appeal to fear": "appeal to emotion",
    "appeal to nature": "fallacy of relevance",
    "false analogy": "fallacy of logic",
    "ad populum": "ad populum",
    "false causality": "false causality",
    "straw man": "fallacy of extension",
    "guilt by association": "ad hominem",
    "appeal to (false) authority": "fallacy of credibility",
    "equivocation": "equivocation",
    "circular reasoning": "circular reasoning",
    "appeal to anger": "appeal to emotion",
    "appeal to worse problems": "fallacy of relevance",
    "appeal to tradition": "fallacy of relevance",
    "fallacy of division": "fallacy of logic",
    "appeal to positive emotion": "appeal to emotion",
    "tu quoque": "ad hominem",
    "appeal to pity": "appeal to emotion",
}

def load_mafalda_docs():
    docs = []
    for line in open(os.path.join(DATA_DIR, "gold_standard_dataset.jsonl"), encoding="utf-8"):
        if not line.strip():
            continue
        rec = json.loads(line)
        swl = rec["sentences_with_labels"]
        if isinstance(swl, str):
            swl = json.loads(swl)
        sents = []
        for sent, labels in swl.items():
            flat = [x for grp in labels for x in (grp if isinstance(grp, list) else [grp])]
            types = [MAFALDA_TO_OURS[t] for t in flat if t != "nothing" and t in MAFALDA_TO_OURS]
            sent = sent.strip()
            if len(sent.split()) >= MIN_WORDS:
                sents.append((sent, types))
        if sents:
            docs.append(sents)
    return docs

rng = np.random.RandomState(SEED)
docs = load_mafalda_docs()

# THREE-way split by document. The val set exists so we can pick the best epoch
# WITHOUT touching the test set — otherwise the reported score is optimistically
# biased (you'd be selecting on the very data you report on).
order = rng.permutation(len(docs))
n_test = int(TEST_FRAC * len(docs))
n_val = int(VAL_FRAC * len(docs))
test_ids  = order[:n_test]
val_ids   = order[n_test:n_test + n_val]
train_ids = order[n_test + n_val:]

maf_train = [s for i in train_ids for s in docs[i]]
maf_val   = [s for i in val_ids   for s in docs[i]]
maf_test  = [s for i in test_ids  for s in docs[i]]
print(f"MAFALDA: {len(docs)} docs -> {len(maf_train)} train / {len(maf_val)} val / {len(maf_test)} test sentences")
print("  (val = pick the best epoch;  test = the honest number we report)")

def det_df(pairs):
    return pd.DataFrame([(s, "fallacy" if ty else "not_fallacy") for s, ty in pairs],
                        columns=["text", "label"])

def typ_df(pairs):
    return pd.DataFrame([(s, ty[0]) for s, ty in pairs if ty], columns=["text", "label"])

# ---- Stage 1: detector (fallacy vs not_fallacy) ----
con = pd.read_csv(os.path.join(DATA_DIR, "contrastive_dataset.csv"))
det_rows  = [(str(t).strip(), "fallacy" if l == "fallacy" else "not_fallacy")
             for t, l in zip(con["text"], con["label"])]
det_train = pd.concat([pd.DataFrame(det_rows, columns=["text", "label"]),
                       det_df(maf_train)]).drop_duplicates().reset_index(drop=True)
det_val, det_test = det_df(maf_val), det_df(maf_test)

# ---- Stage 2: typer (which fallacy) ----
edu = pd.read_csv(os.path.join(DATA_DIR, "edu_train.csv"))[["source_article", "updated_label"]]
edu.columns = ["text", "label"]
edu["text"] = edu["text"].astype(str).str.strip()
typ_train = pd.concat([edu, typ_df(maf_train)]).drop_duplicates().reset_index(drop=True)
typ_val, typ_test = typ_df(maf_val), typ_df(maf_test)

print(f"\ndetector: {len(det_train)} train / {len(det_val)} val / {len(det_test)} test")
print(det_train["label"].value_counts().to_string())
print(f"\ntyper: {len(typ_train)} train / {len(typ_val)} val / {len(typ_test)} test"
      f" | {typ_train['label'].nunique()} classes")
print(typ_train["label"].value_counts().to_string())

assert len(det_val) and len(det_test) and len(typ_val) and len(typ_test), "a split came out empty"

## 3. Fine-tuning

Trained with class weights (some classes are rare), and evaluated **every epoch on the
held-out real-world test set** — the only honest measure of "does this work on real prose?".
We keep the epoch with the best real-world macro-F1.

In [ ]:
class DS(Dataset):
    def __init__(self, texts, labels, tok):
        enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding="max_length",
                  return_token_type_ids=False, return_tensors="pt")
        self.ids, self.mask = enc["input_ids"], enc["attention_mask"]
        self.y = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i):
        return {"input_ids": self.ids[i], "attention_mask": self.mask[i], "labels": self.y[i]}

def _loader(df, cmap, tok, shuffle=False):
    return DataLoader(DS(df["text"], df["label"].map(cmap), tok), batch_size=BATCH, shuffle=shuffle)


def _predict(model, loader):
    model.eval(); preds, gold = [], []
    with torch.no_grad():
        for b in loader:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            preds += model(input_ids=b["input_ids"],
                           attention_mask=b["attention_mask"]).logits.argmax(1).cpu().tolist()
            gold += b["labels"].cpu().tolist()
    return gold, preds


def train_stage(name, tr, va, te, out_dir, epochs):
    """Train, pick the best epoch on VAL, then report the honest score on TEST."""
    classes = sorted(tr["label"].unique())
    cmap = {c: i for i, c in enumerate(classes)}
    va = va[va["label"].isin(cmap)]
    te = te[te["label"].isin(cmap)]

    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(classes)).to(DEVICE)
    tl = _loader(tr, cmap, tok, shuffle=True)
    vl = _loader(va, cmap, tok)
    sl = _loader(te, cmap, tok)

    counts = tr["label"].map(cmap).value_counts().sort_index()
    w = torch.tensor([len(tr) / (len(classes) * counts.get(i, 1)) for i in range(len(classes))],
                     dtype=torch.float).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    print(f"\n=== {name}: {len(tr)} train / {len(va)} val / {len(te)} test | {len(classes)} classes ===")
    best_f1, best_state = -1.0, None
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time()
        for b in tl:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            loss = crit(model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits,
                        b["labels"])
            opt.zero_grad(); loss.backward(); opt.step()

        g, p = _predict(model, vl)                      # <-- selection on VAL only
        f1 = f1_score(g, p, average="macro", zero_division=0)
        flag = ""
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            flag = "  <- best so far"
        print(f"  epoch {ep}: val macro-F1 = {f1:.3f}   ({time.time()-t0:.0f}s){flag}")

    model.load_state_dict(best_state)

    # ---- the honest number: TEST set, never used for any decision ----
    gold, preds = _predict(model, sl)
    print(f"\n  >> HELD-OUT REAL-WORLD TEST (best epoch, val F1 {best_f1:.3f}) <<")
    print(f"  test macro-F1 = {f1_score(gold, preds, average='macro', zero_division=0):.3f}")
    print(classification_report(gold, preds, labels=range(len(classes)),
                                target_names=classes, zero_division=0, digits=2))
    if "not_fallacy" in cmap:
        cm = confusion_matrix(gold, preds, labels=[cmap["not_fallacy"], cmap["fallacy"]])
        tn, fp = cm[0]
        print(f"  false-alarm rate on clean prose: {fp/max(tn+fp,1):.0%}  ({fp}/{tn+fp})")
        print("  (the OLD detector was 62% — that was the bug we're fixing)")

    out = os.path.join(OUT_DIR, out_dir); os.makedirs(out, exist_ok=True)
    model.save_pretrained(out); tok.save_pretrained(out)
    json.dump({"classes": classes, "version": DATA_VERSION},
              open(os.path.join(out, "classes.json"), "w"), indent=2)
    print(f"  -> saved {out_dir}")
    return classes

### Stage 1 — Detector (the one that was broken)

In [ ]:
det_classes = train_stage("DETECTOR", det_train, det_val, det_test, "detector_distilbert", DET_EPOCHS)

### Stage 2 — Typer

In [ ]:
typ_classes = train_stage("TYPER", typ_train, typ_val, typ_test, "typer_distilbert", TYP_EPOCHS)

## 4. Write pipeline metadata

In [ ]:
meta = {"detector_model": "bert", "typer_model": "bert",
        "detector_classes": det_classes, "typer_classes": typ_classes,
        "max_len": MAX_LEN, "data_version": DATA_VERSION}
json.dump(meta, open(os.path.join(OUT_DIR, "pipeline.json"), "w"), indent=2)
print(json.dumps(meta, indent=2)[:400])
print("\ncontents:", sorted(os.listdir(OUT_DIR)))

## 5. Sanity check — try it on real debate lines

The old model called *"Thank you both."* a fallacy with 0.99 confidence. The new one should
be quiet on the clean lines and confident on the real fallacies.

In [ ]:
det_tok = AutoTokenizer.from_pretrained(os.path.join(OUT_DIR, "detector_distilbert"))
det_mdl = AutoModelForSequenceClassification.from_pretrained(os.path.join(OUT_DIR, "detector_distilbert")).to(DEVICE).eval()
typ_tok = AutoTokenizer.from_pretrained(os.path.join(OUT_DIR, "typer_distilbert"))
typ_mdl = AutoModelForSequenceClassification.from_pretrained(os.path.join(OUT_DIR, "typer_distilbert")).to(DEVICE).eval()

def probs(tok, mdl, classes, text):
    enc = tok(text, truncation=True, max_length=MAX_LEN, return_token_type_ids=False,
              return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        p = torch.softmax(mdl(**enc).logits[0], dim=-1)
    return {c: float(p[i]) for i, c in enumerate(classes)}

samples = [
    ("CLEAN", "Thank you both."),
    ("CLEAN", "The Congressional Budget Office found the reform would reduce the deficit by 200 billion over ten years."),
    ("CLEAN", "South Korea and France built fleets on schedule and at cost by standardizing designs."),
    ("FALLACY (false dilemma)", "Either we build firm clean capacity, or we keep burning fossil fuels to back up the intermittent stuff."),
    ("FALLACY (ad hominem)",    "Interesting that the person telling us to bet the grid on nuclear spent six years consulting for a reactor vendor."),
    ("FALLACY (hasty gen.)",    "Two people on my street died of cancer within a few years of each other, and I don't think I'm wrong to generalize from it."),
    ("FALLACY (authority)",     "Even a Nobel laureate has said nuclear is essential, and when a mind like that weighs in, the debate is largely settled."),
]
print(f"{'expected':26s} {'P(fallacy)':>11}  predicted type")
print("-"*78)
for tag, s in samples:
    pf = probs(det_tok, det_mdl, det_classes, s).get("fallacy", 0)
    tp = probs(typ_tok, typ_mdl, typ_classes, s)
    top = max(tp, key=tp.get)
    shown = f"{top} ({tp[top]:.2f})" if pf >= 0.70 else "-"
    print(f"{tag:26s} {pf:11.2f}  {shown}")

## 6. Download the models — **one at a time**

⚠️ **Do not zip both models together.** A combined zip is ~500 MB and Colab's
`files.download()` silently truncates large files. A zip is written sequentially, so a cut
partway through leaves you with a corrupt first model and a *completely missing* second one
— which looks like "training didn't finish" but is really just a failed download.

So we make **two separate zips (~250 MB each)** and print the exact byte size of each.
**Check the downloaded file matches the size printed here** before using it.

In [ ]:
import shutil, os

sizes = {}
for stage in ["detector_distilbert", "typer_distilbert"]:
    src = os.path.join(OUT_DIR, stage)
    assert os.path.exists(os.path.join(src, "model.safetensors")), f"{stage} did not train!"
    z = shutil.make_archive(f"/content/{stage}", "zip", root_dir=OUT_DIR, base_dir=stage)
    sizes[z] = os.path.getsize(z)
    print(f"{os.path.basename(z):28s} {sizes[z]:,} bytes  ({sizes[z]/1e6:.0f} MB)")

# pipeline.json is tiny — bundle it with nothing else
with open(os.path.join(OUT_DIR, "pipeline.json")) as fh:
    print("\npipeline.json:\n", fh.read())

print("\n>>> VERIFY each downloaded .zip matches the byte size above. If it doesn't,")
print(">>> re-run this cell (or use the Google Drive route in the next cell).")

In [ ]:
from google.colab import files
for stage in ["detector_distilbert", "typer_distilbert"]:
    files.download(f"/content/{stage}.zip")
files.download(os.path.join(OUT_DIR, "pipeline.json"))

### More reliable: copy to Google Drive instead

If a browser download truncates again, push the models to Drive and download them from
there — Drive supports resuming, so large files arrive intact.

In [ ]:
# OPTIONAL — run only if the direct downloads keep truncating.
from google.colab import drive
drive.mount("/content/drive")

dest = "/content/drive/MyDrive/fallacy_models_v2"
shutil.rmtree(dest, ignore_errors=True)
shutil.copytree(OUT_DIR, dest)
print("copied to Drive:", dest)
for root, _, fs in os.walk(dest):
    for f in fs:
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, dest):48s} {os.path.getsize(p):,} bytes")

---
## After downloading

```powershell
# unzip fallacy_models_v2.zip -> it contains two_stage/
# copy its contents into fallacysuspect\models\two_stage\  (overwrite)

cd C:\Users\Absol\OneDrive\Documents\GitHub\PortFol\fallacysuspect
python -m fallacy_warn serve
```

Paste the nuclear debate. The **false dilemma** ("Either we build firm clean capacity, or…")
should now be caught — the old detector rated it only 0.41 and missed it entirely.

If it still misbehaves, tune the gates without retraining:
```powershell
$env:FALLACY_DETECT_THRESHOLD = "0.80"   # stricter -> fewer flags
$env:FALLACY_TYPE_THRESHOLD   = "0.55"
```